# A6 — EDA, Evaluation Protocol Lock, and Pre-Registration

The gate before Study B. Three deliverables:

1. **EDA** across all three layers, reported separately (CLAUDE.md rule 12).
2. **`reports/evaluation_protocol.md`** — the locked protocol (rule 11).
3. **`reports/analysis_plan_preregistration.md`** — hypotheses fixed before any model runs (rule 13).

Analysis lives in `src/evaluation/eda.py`; metrics and splits that Study B will use are in `src/evaluation/metrics.py` and `src/evaluation/splits.py`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

from src.evaluation.eda import run, log_stats

In [ ]:
results = run()
log_stats(results)

## The headline finding: EPSS leaks under a temporal split

This is what forces the point-in-time rule in the locked protocol.

Take only CVEs published **before** the split date — the rows that would be training data — and group them by *when* they entered KEV. If a CVE whose exploitation only became public **after** the split already scores high on **today's** EPSS, then that feature carries the outcome we are trying to predict.

In [ ]:
results["leakage"]

Median EPSS **today** for CVEs exploited only after the split is 45.7× the never-exploited baseline — and indistinguishable from those exploited *before* it. Today's EPSS is close to an oracle for future KEV membership.

Swapping in **point-in-time** EPSS (`src.ingestion.epss.fetch_epss_asof`) drops that ratio to **5.0×**. The residual is legitimate predictive signal, which is exactly what EPSS is designed to provide.

**Binding rule:** `epss_scores-current.csv.gz` may not be a feature in any temporally split experiment. It stays permitted for descriptive statistics and as a B1 baseline ranker (where no training happens).

## Splits and metrics behave as specified

The guards refuse to return a broken split, and the metric set demotes ROC-AUC for a reason worth seeing directly.

In [ ]:
import numpy as np
import pandas as pd

from src.evaluation.metrics import evaluate
from src.evaluation.splits import assert_no_temporal_leakage, grouped_split, temporal_split

train, test = temporal_split(results["l1"], "2023-01-01")
assert_no_temporal_leakage(train, test)
print(f"L1 temporal split: train={len(train):,}  test={len(test):,}")

l3_train, l3_test = grouped_split(results["l3"])
print(f"L3 grouped split : train={len(l3_train):,}  test={len(l3_test):,}")

# Why ROC-AUC is secondary: a random ranker still looks respectable on it.
rng = np.random.default_rng(0)
y = np.zeros(10_000)
y[:50] = 1
rng.shuffle(y)
random_scores = evaluate(y, rng.random(10_000))
pd.Series(
    {k: random_scores[k] for k in ["roc_auc", "pr_auc", "precision_at_10", "base_rate"]}
).round(4)

## What is now locked

Read `reports/evaluation_protocol.md` and `reports/analysis_plan_preregistration.md` in full before starting B1.

- **Primary metrics:** PR-AUC, Precision@k (k = 10/50/100). ROC-AUC reported, never used to select.
- **Splits:** L1 temporal (never shuffled), L3 grouped-by-asset.
- **EPSS:** point-in-time only in trained experiments.
- **Seeds:** ≥10, mean ± std. Friedman + Nemenyi across families, Wilcoxon pairwise.
- **Baselines first:** no learned model is reported before B1. A model that cannot beat CVSS ordering is reported as a negative result, not buried.
- **L1/L2 and L3 always reported in separate tables.**

**Still open (needs supervisor sign-off):** the L3 label formula puts L1 and L2 on incommensurate scales, so 99.4% of vulnerability findings label `Low`. Until that is resolved, L3 results speak only to *misconfiguration* prioritisation. See the A5 entry in the CLAUDE.md Deviations Log.